# BirdCLEF+ 2025 - Pretrained models: original A1 sigmoid probabilities on ALL train_soundscapes (teacher for a noisy student)

**T4 GPU, internet off, no training.** The teacher is the original, unadapted A1 model (`sc4000-birdclef-m3-a1-v2s-train`) in full eval mode, float32, no AMP. The files are `sorted(train_soundscapes/*.ogg)`, counted at runtime, and `test_soundscapes` is never read.

**Windows:** non-overlapping 5 s, `n_win = max(1, ceil(len / WIN))`, end padded with zeros, `win` 0-based. This is the same rule and sort as in `ns_perch_all`.

**Preprocessing:** the A1 fast-inference `to_input` and `windows_to_inputs`, unchanged. `decode` also applies the failure and resampling rules of `ns_perch_all`, which change nothing for a normal 32 kHz file.

**Decode failures** are recorded in the meta file, and more than 20 stop the run. Each notebook keeps its own list, so the later student must assert that the lists and the two csv files are equal.

**Output:** `a1_all_probs.npy` (float32, N x 206), `soundscape_all_windows.csv` (filename, win), `species.json`, `ns_a1_meta.json`. The final names only appear after the check against the saved `a2-a1distill` teacher probabilities has passed, and the meta file is written last.

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"                # set before numpy loads; each worker uses one core

import os, time
import torch

T_NOTEBOOK = time.time()
SMOKE = os.environ.get("M3_SMOKE_TEST") == "1"     # local CPU test only, never set on Kaggle
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("torch.cuda.get_device_name(0):", torch.cuda.get_device_name(0))
elif SMOKE:
    print("M3_SMOKE_TEST=1: no GPU, running the local CPU test")
else:
    raise RuntimeError("No GPU found. Enable the T4 accelerator; this notebook never falls back to CPU.")

In [ ]:
import collections, gc, glob, importlib, json, multiprocessing as mp, sys, tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import timm

# On Kaggle every check is on. M3_STRICT only re-enables named checks inside a local smoke test.
STRICT = set(filter(None, os.environ.get("M3_STRICT", "").split(",")))
def strict(name):
    return not SMOKE or name in STRICT

OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (smoke test)"
MAX_FAILED = int(os.environ.get("M3_MAX_FAILED", "20"))
GUARD_AFTER = int(os.environ.get("M3_GUARD_WINDOWS", "500"))
RECHECK_EVERY = int(os.environ.get("M3_GUARD_RECHECK", "5000"))   # read-ahead hides slow decoding at the first check
REF_WINDOWS = 12000
SANITY_LIMIT = 1e-3

def find_dir(marker, env):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    if len(hits) != 1:
        raise FileNotFoundError(f"expected exactly one {marker} under /kaggle/input, found {hits}")
    return Path(hits[0]).parent

def list_soundscapes(data_dir):
    """The file order of both notebooks: bare names of train_soundscapes/*.ogg, sorted."""
    names = sorted(p.name for p in (data_dir / "train_soundscapes").glob("*.ogg"))
    if not names:
        raise RuntimeError(f"no .ogg files in {data_dir / 'train_soundscapes'}")
    return names

def add_failure(failed, name, err):
    failed.append((name, err))
    print("decode failure:", name, err, flush=True)
    if len(failed) > MAX_FAILED:
        raise RuntimeError(f"{len(failed)} files failed to decode (limit {MAX_FAILED}), first: {failed[:3]}")

def make_guard(label, total_files):
    """Projects the whole notebook after GUARD_AFTER windows and again every RECHECK_EVERY windows. The rate starts
    after the first GPU batch, so one-off graph compilation does not inflate it."""
    state = dict(mark=None, next_check=GUARD_AFTER)
    def check(n_done, windows_seen, files_ok):
        now = time.time()
        if state["mark"] is None:
            state["mark"] = (n_done, now)
            return
        n0, t0 = state["mark"]
        if n_done - n0 < state["next_check"]:
            return
        state["next_check"] = n_done - n0 + RECHECK_EVERY
        rate = (n_done - n0) / max(now - t0, 1e-6)
        est_total = windows_seen / max(files_ok, 1) * total_files
        elapsed = now - T_NOTEBOOK
        projected = elapsed + max(est_total - n_done, 0) / rate
        msg = (f"{label}: {rate:.1f} windows/s after {n_done} windows, about {est_total:.0f} windows in total, "
               f"{elapsed:.0f}s elapsed -> projected notebook time {projected:.0f}s = {projected / 3600:.2f} h "
               f"(limit {BUDGET_S:.0f}s = {BUDGET_S / 3600:.2f} h)")
        print("TIMING GUARD |", msg, flush=True)
        if projected > BUDGET_S:
            raise RuntimeError(f"Projected GPU time is over the limit. {msg}")
    return check

def load_reference_keys(csv_path, n_rows, dim, known_names):
    """Checks that cost nothing, before any GPU work."""
    ref = pd.read_csv(csv_path)
    if list(ref.columns[:2]) != ["filename", "win"]:
        raise RuntimeError(f"unexpected reference csv columns: {list(ref.columns)}")
    ref = ref[["filename", "win"]].astype({"filename": str, "win": int})
    if ref.duplicated().any():
        raise RuntimeError("duplicate (filename, win) rows in the reference csv")
    if len(ref) != n_rows:
        raise RuntimeError(f"reference csv has {len(ref)} rows, its array has {n_rows}")
    if strict("ref") and n_rows != REF_WINDOWS:
        raise RuntimeError(f"expected {REF_WINDOWS} reference windows, found {n_rows}")
    unknown = sorted(set(ref["filename"]) - set(known_names))
    if unknown:
        raise RuntimeError(f"{len(unknown)} reference files are not in train_soundscapes, first: {unknown[:3]}")
    print(f"reference: {len(ref)} windows, {ref['filename'].nunique()} files, {dim}-d", flush=True)
    return ref

def align_to_reference(ref, windows):
    """Row of the new array for each reference row, matched on (filename, win), in reference row order."""
    merged = ref.merge(windows.assign(new_row=np.arange(len(windows))), on=["filename", "win"], how="left", validate="one_to_one")
    missing = merged[merged["new_row"].isna()]
    if len(missing):
        raise RuntimeError(f"{len(missing)} reference windows have no row in this run, first: "
                           f"{missing[['filename', 'win']].head(3).values.tolist()}")
    return merged["new_row"].astype(np.int64).values

def check_windows_table(windows, names, failed):
    """Sorted file order, windows 0..n-1 per file, and exactly the failed files left out."""
    done = list(dict.fromkeys(windows["filename"].tolist()))
    bad = {n for n, _ in failed}
    if done != [n for n in names if n not in bad]:
        raise RuntimeError("the embedded files are not the sorted file list minus the failed files")
    for name, g in windows.groupby("filename", sort=False):
        assert g["win"].tolist() == list(range(len(g))), f"windows of {name} are not 0..n-1"

def finish_array(bin_path, npy_tmp, n_rows, dim, chunk_rows=4096):
    """Turns the appended raw float32 file into a .npy in chunks, checks every value, and returns its name."""
    expected = n_rows * dim * 4
    if os.path.getsize(bin_path) != expected:
        raise RuntimeError(f"{bin_path} has {os.path.getsize(bin_path)} bytes, expected {expected}")
    mm = np.lib.format.open_memmap(npy_tmp, mode="w+", dtype=np.float32, shape=(n_rows, dim))
    with open(bin_path, "rb") as f:
        for start in range(0, n_rows, chunk_rows):
            part = np.frombuffer(f.read(min(chunk_rows, n_rows - start) * dim * 4), dtype="<f4").reshape(-1, dim)
            if not np.isfinite(part).all():
                raise RuntimeError(f"non-finite values in rows {start}..{start + len(part)}")
            mm[start:start + len(part)] = part
    mm.flush()
    del mm
    gc.collect()
    os.remove(bin_path)

def sanity_compare(npy_tmp, rows, ref_values, label):
    """Max and mean absolute difference between the new rows and the reference rows. The raise is left to the caller."""
    new = np.load(npy_tmp, mmap_mode="r")
    picked = np.asarray(new[rows], dtype=np.float32)
    del new
    gc.collect()
    assert picked.shape == ref_values.shape, (picked.shape, ref_values.shape)
    diff = np.abs(picked - ref_values)
    result = dict(n_rows=int(len(rows)), max_abs_diff=float(diff.max()), mean_abs_diff=float(diff.mean()), limit=SANITY_LIMIT)
    print(f"SANITY | {label}: {result['n_rows']} rows | max abs diff {result['max_abs_diff']:.3e} | "
          f"mean abs diff {result['mean_abs_diff']:.3e} | limit {SANITY_LIMIT}", flush=True)
    return result

def publish(tmp, final):
    os.replace(tmp, OUT / final)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
NUM_WORKERS = int(os.environ.get("M3_WORKERS", "4"))
BATCH = int(os.environ.get("M3_BATCH_SIZE", "64"))
CHUNK_FILES = int(os.environ.get("M3_CHUNK_FILES", "32"))      # 384 windows per chunk for 1-minute files
MAX_IN_FLIGHT = 3                                              # chunks decoded ahead of the GPU; bounds the RAM used
BUDGET_S = float(os.environ.get("M3_BUDGET_S", 1.0 * 3600))    # whole notebook
TEACHER_KERNEL = "sc4000-birdclef-m3-a1-v2s-train"             # the original A1, never the a1-adabn kernel

def find_model_dir(kernel):
    if os.environ.get("M3_MODEL_DIR"):      # local tests only
        return Path(os.environ["M3_MODEL_DIR"])
    hits = [h for h in glob.glob("/kaggle/input/**/model.pth", recursive=True) if kernel in h]
    if len(hits) != 1:
        raise RuntimeError(f"expected one model.pth under kernel {kernel}, found {hits}")
    return Path(hits[0]).parent

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR")
MODEL_DIR = find_model_dir(TEACHER_KERNEL)
REF_DIR = find_dir("teacher_a1_probs.npy", "M3_REF_DIR")               # output of a2-a1distill
KEY_DIR = find_dir("soundscape_windows.csv", "M3_KEY_DIR")             # a2-a1distill saved its array in the row order of this file
print("Data:", DATA, "| teacher:", MODEL_DIR, "| reference:", REF_DIR, "| keys:", KEY_DIR, "| CPUs:", os.cpu_count(), "| smoke:", SMOKE)

A1_CFG = json.load(open(MODEL_DIR / "cfg.json"))
species = json.load(open(MODEL_DIR / "species.json"))
submission_species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
print("A1 cfg norm:", A1_CFG.get("norm"), "| model:", A1_CFG["model"], "| classes:", len(species))
assert A1_CFG.get("norm") == "zscore_minmax", f"expected zscore_minmax, cfg.json has {A1_CFG.get('norm')}"
assert A1_CFG["sr"] == 32000 and A1_CFG["duration"] == 5, (A1_CFG["sr"], A1_CFG["duration"])
if not A1_CFG["model"].startswith("tf_efficientnetv2_s"):
    raise RuntimeError(f"expected tf_efficientnetv2_s, cfg.json has {A1_CFG['model']}")
assert species == submission_species, "A1 species list differs from sample_submission"
N_CLASSES = len(species)
if strict("classes") and N_CLASSES != 206:
    raise RuntimeError(f"expected 206 classes, found {N_CLASSES}")

## File list and reference (before any GPU work)

In [ ]:
names = list_soundscapes(DATA)
paths = [DATA / "train_soundscapes" / n for n in names]
print(len(paths), "soundscapes in train_soundscapes")

# a2-a1distill saves raw sigmoid probabilities in this file (before its power and floor), so they are compared directly.
ref_values = np.load(REF_DIR / "teacher_a1_probs.npy").astype(np.float32)
if ref_values.ndim != 2 or ref_values.shape[1] != N_CLASSES:
    raise RuntimeError(f"reference probabilities have shape {ref_values.shape}")
ref_species = json.load(open(REF_DIR / "species.json"))
assert ref_species == species, "the a2-a1distill species list differs from A1's"
ref_keys = load_reference_keys(KEY_DIR / "soundscape_windows.csv", len(ref_values), N_CLASSES, names)

## Worker code
`to_input` and `windows_to_inputs` are the A1 fast-inference functions, used unchanged.

In [ ]:
LIB_SRC = r"""
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
from scipy.signal import resample_poly

CFG, WIN, TOP_DB = None, None, None

class EmptyAudioError(Exception):
    pass

class NonFiniteAudioError(Exception):
    pass

# Only these count as a decode failure of the file. Anything else is a bug and stops the notebook.
DECODE_ERRORS = (getattr(sf, "SoundFileError", RuntimeError), EmptyAudioError, NonFiniteAudioError)

def configure(cfg):
    global CFG, WIN, TOP_DB
    CFG, WIN, TOP_DB = cfg, cfg["sr"] * cfg["duration"], float(cfg["norm_top_db"])

def to_input(db):
    # Same as to_input in the A1 notebooks. For norm "v2" this equals M1's (db + 80) / 80.
    db = np.maximum(db - db.max(), -TOP_DB)
    if CFG["norm"] == "zscore_minmax":
        x = (db - db.mean()) / (db.std() + 1e-6)
        return ((x - x.min()) / (x.max() - x.min() + 1e-6)).astype(np.float32)
    return ((db + TOP_DB) / TOP_DB).astype(np.float32)

def decode(path):
    # The fast-inference decode plus the rules of ns_perch_all (sample rate, empty and non-finite files).
    y, sr = sf.read(str(path), dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != CFG["sr"]:
        y = resample_poly(y, CFG["sr"], sr).astype(np.float32)
    if len(y) == 0:
        raise EmptyAudioError(f"no samples in {Path(path).name}")
    if not np.isfinite(y).all():
        raise NonFiniteAudioError(f"non-finite samples in {Path(path).name}")
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    return y.reshape(n_win, WIN)

def windows_to_inputs(batch):
    # One melspectrogram call for the file. Each row gets its own STFT, as in the per-window loop.
    power = librosa.feature.melspectrogram(
        y=batch, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])
    # ref=np.max and the normalisation stay per window.
    return np.stack([to_input(librosa.power_to_db(m, ref=np.max, top_db=TOP_DB)) for m in power])

def file_windows(path):
    # Only the decode can record a failure; an error in the mel code is a bug and stops the notebook.
    try:
        y = decode(path)
    except DECODE_ERRORS as e:
        return None, repr(e)
    return windows_to_inputs(y), ""
"""
LIB_DIR = Path(tempfile.mkdtemp(prefix="ns_a1_lib_"))     # outside the output folder
(LIB_DIR / "ns_a1_lib.py").write_text(LIB_SRC)
sys.path.insert(0, str(LIB_DIR))
importlib.invalidate_caches()
import ns_a1_lib as lib
lib.configure(A1_CFG)

## Original A1 on every window

In [ ]:
t_start = time.time()
teacher = timm.create_model(A1_CFG["model"], pretrained=False, in_chans=1, num_classes=N_CLASSES)
teacher.load_state_dict(torch.load(MODEL_DIR / "model.pth", map_location="cpu"))
teacher = teacher.to(DEVICE).eval()
assert not any(m.training for m in teacher.modules()), "teacher is not in full eval mode"

BIN_TMP = OUT / "a1_all_probs.tmp.bin"
NPY_TMP = OUT / "a1_all_probs.tmp.npy"
guard = make_guard("A1", len(paths))
row_j, row_w, failed = [], [], []
n_done, n_seen, files_ok = 0, 0, 0
t_wait, t_model = 0.0, 0.0

def consume(first_j, results, sink):
    """One chunk from the Pool, in file order: record failures, then run the model and append the probabilities."""
    global n_done, n_seen, files_ok, t_model
    ok = []
    for j, (w, err) in enumerate(results, first_j):
        if err:
            add_failure(failed, names[j], err); continue
        row_j.extend([j] * len(w)); row_w.extend(range(len(w)))
        ok.append(w); n_seen += len(w); files_ok += 1
    if not ok:
        return
    x = np.concatenate(ok)
    assert x.dtype == np.float32 and x.ndim == 3, (x.dtype, x.shape)
    for i in range(0, len(x), BATCH):
        t = time.time()
        xb = torch.from_numpy(x[i:i + BATCH])[:, None].to(DEVICE)
        assert xb.dtype == torch.float32
        with torch.inference_mode():
            p = torch.sigmoid(teacher(xb).float()).cpu().numpy()    # .cpu() waits for the GPU, so the time is real
        t_model += time.time() - t
        assert p.shape == (len(xb), N_CLASSES), p.shape
        if not (np.isfinite(p).all() and p.min() >= 0 and p.max() <= 1):
            raise RuntimeError("probabilities out of range or non-finite")
        sink.write(p.astype("<f4").tobytes())
        n_done += len(xb)
        guard(n_done, n_seen, files_ok)

# Spawn: the parent has already touched CUDA. Chunks are submitted ahead and consumed in order, at most MAX_IN_FLIGHT at a
# time, so the decoded windows held in RAM stay bounded however far the GPU lags behind.
chunks = [paths[s:s + CHUNK_FILES] for s in range(0, len(paths), CHUNK_FILES)]
in_flight = collections.deque()
with open(BIN_TMP, "wb") as sink, mp.get_context("spawn").Pool(NUM_WORKERS, initializer=lib.configure, initargs=(A1_CFG,)) as pool:
    for c, chunk in enumerate(chunks):
        in_flight.append((c * CHUNK_FILES, pool.map_async(lib.file_windows, chunk, chunksize=1)))
        while len(in_flight) >= MAX_IN_FLIGHT or (c == len(chunks) - 1 and in_flight):
            first_j, res = in_flight.popleft()
            t = time.time()
            results = res.get()
            t_wait += time.time() - t
            consume(first_j, results, sink)
            del results
        if c % 25 == 0:
            done = min((c + 1) * CHUNK_FILES, len(paths)); el = time.time() - t_start
            print(f"A1 {done}/{len(paths)} files submitted | {n_done} windows | {el:.0f}s | wait {t_wait:.0f}s | model {t_model:.0f}s", flush=True)

if DEVICE == "cuda":
    torch.cuda.synchronize()
assert not any(m.training for m in teacher.modules()), "teacher left eval mode"
a1_wall = time.time() - t_start
print(f"A1 done: {n_done} windows | failed decodes {len(failed)} | wall {a1_wall:.0f}s | waiting for windows {t_wait:.0f}s | model {t_model:.0f}s")
del teacher
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## Checks against the `a2-a1distill` teacher probabilities, then atomic publication

In [ ]:
windows = pd.DataFrame({"filename": [names[j] for j in row_j], "win": row_w})
N = len(windows)
assert N == n_done == n_seen, (N, n_done, n_seen)
if N == 0:
    raise RuntimeError("no windows were processed")
check_windows_table(windows, names, failed)

finish_array(BIN_TMP, NPY_TMP, N, N_CLASSES)
rows = align_to_reference(ref_keys, windows)
sanity = sanity_compare(NPY_TMP, rows, ref_values, "A1 vs a2-a1distill teacher (raw probabilities)")
if sanity["max_abs_diff"] > SANITY_LIMIT:
    # The temporary array stays in the output for diagnosis; no final name exists, so nothing downstream can use it.
    raise RuntimeError(f"Sanity check failed: max abs diff {sanity['max_abs_diff']:.3e} > {SANITY_LIMIT} "
                       f"(mean {sanity['mean_abs_diff']:.3e}, {sanity['n_rows']} rows)")

per_file = windows.groupby("filename", sort=False).size()
wall_total = time.time() - T_NOTEBOOK
meta = dict(
    teacher=f"{TEACHER_KERNEL}/model.pth, unadapted, full eval, float32, no AMP", batch_size=BATCH, device_name=GPU_NAME,
    file_order="sorted(train_soundscapes/*.ogg) by bare filename",
    files_listed=len(names), files_processed=int(len(per_file)), windows=N,
    windows_per_file_counts={str(int(k)): int(v) for k, v in per_file.value_counts().items()},
    failed_files=[dict(filename=n, error=e) for n, e in failed], sanity_vs_a2_a1distill=sanity,
    wall_s=dict(a1=a1_wall, waiting_for_windows=t_wait, notebook=wall_total), model_s=t_model)

csv_tmp = OUT / "soundscape_all_windows.tmp.csv"
windows.to_csv(csv_tmp, index=False, lineterminator="\n")
species_tmp = OUT / "species.tmp.json"
with open(species_tmp, "w") as f:
    json.dump(species, f)
publish(NPY_TMP, "a1_all_probs.npy")
publish(csv_tmp, "soundscape_all_windows.csv")
publish(species_tmp, "species.json")
meta_tmp = OUT / "ns_a1_meta.tmp.json"             # written last: its presence means the run is complete
with open(meta_tmp, "w") as f:
    json.dump(meta, f, indent=1)
publish(meta_tmp, "ns_a1_meta.json")

print("Windows per file:", meta["windows_per_file_counts"], "| failed files:", len(failed))
print(f"TIMING | {N} windows, {len(per_file)} files | A1 wall {a1_wall:.0f}s, {N / a1_wall:.1f} windows/s | "
      f"waiting for windows {t_wait:.0f}s | model {t_model:.0f}s | notebook {wall_total:.0f}s = {wall_total / 3600:.2f} h | device {GPU_NAME}")